# Multi-dialect Arabic ASR: Whisper-small fine-tuning (Phase 2)

Reimplementation of Ozyilmaz et al., *Overcoming Data Scarcity in Multi-Dialectal Arabic ASR via Whisper
Fine-Tuning* (Interspeech 2025) with Paper 2-style Arabic text normalisation. Run the cells top to bottom on a
**GPU runtime** (Colab: Runtime > Change runtime type > T4 GPU; Kaggle: Accelerator > GPU, Internet on).

* Training commands are **resumable**: if the session dies, re-run the setup cells, then the same training cell.
* Checkpoints and results go to Google Drive (Colab) or `/kaggle/working` (Kaggle; use *Save Version > Save & Run All*
  for long runs so outputs persist).
* Details: `docs/EXPERIMENTS.md` (what each step answers) and `docs/PREPROCESSING.md` (text/audio pipeline).

## 0. Settings

In [ ]:
# Where the code comes from: a GitHub URL, or a zip of the repository uploaded to Google Drive.
# The repository is public, so nothing else is needed. Only if it becomes private: add a GitHub fine-grained
# token as a secret named GITHUB_TOKEN (Colab: key icon in the left sidebar; Kaggle: Add-ons > Secrets).
REPO_URL = "https://github.com/AhmadYasserHamad/Speech-Processing.git"   # set to "" to use CODE_ZIP instead
CODE_ZIP = "/content/drive/MyDrive/arabic-asr/Speech-Processing.zip"  # used when REPO_URL is empty (Colab)
HF_USERNAME = ""                                                 # for pushing trained models (optional)

## 1. Environment

In [ ]:
import os, sys, subprocess, pathlib
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle")
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content"
    os.environ["ASR_CACHE"] = "/content/cache"                              # fast local disk
    os.environ["ASR_RUNS"] = "/content/drive/MyDrive/arabic-asr/runs"       # survives disconnects
    RESULTS = "/content/drive/MyDrive/arabic-asr/results"
elif ON_KAGGLE:
    WORK = "/kaggle/working"
    os.environ["ASR_CACHE"] = "/tmp/cache"
    os.environ["ASR_RUNS"] = "/kaggle/working/runs"
    RESULTS = "/kaggle/working/results"
else:
    WORK = os.getcwd()
    os.environ.setdefault("ASR_CACHE", "cache"); os.environ.setdefault("ASR_RUNS", "runs")
    RESULTS = "results"
os.environ["RESULTS"] = RESULTS
os.environ["TOKENIZERS_PARALLELISM"] = "false"
pathlib.Path(os.environ["ASR_RUNS"]).mkdir(parents=True, exist_ok=True)
pathlib.Path(RESULTS).mkdir(parents=True, exist_ok=True)
print(os.environ["ASR_CACHE"], os.environ["ASR_RUNS"], RESULTS)

In [ ]:
PROJECT = f"{WORK}/Speech-Processing"

def _github_token():
    """GITHUB_TOKEN from Colab or Kaggle secrets, or None (public repo / zip)."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None

def _git(*args, url=None):
    """Run git with an authenticated URL without ever printing the token."""
    token = _github_token() if url else None
    auth_url = url.replace("https://", f"https://x-access-token:{token}@") if token else url
    cmd = ["git", *[auth_url if a == "<URL>" else a for a in args]]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        raise SystemExit(f"git {args[0]} failed (private repo? check the GITHUB_TOKEN secret):\n{message}")

if not os.path.exists(f"{PROJECT}/pyproject.toml"):
    if REPO_URL:
        _git("clone", "<URL>", PROJECT, url=REPO_URL)
        _git("-C", PROJECT, "remote", "set-url", "origin", REPO_URL)  # keep the token out of .git/config
    elif os.path.exists(CODE_ZIP):
        subprocess.run(["unzip", "-q", "-o", CODE_ZIP, "-d", WORK], check=True)
    else:
        raise SystemExit("Set REPO_URL or upload the repository zip to CODE_ZIP")
elif REPO_URL and os.path.exists(f"{PROJECT}/.git"):
    _git("-C", PROJECT, "pull", "--ff-only", "<URL>", "main", url=REPO_URL)  # same session: get latest code
os.chdir(PROJECT)
print(os.getcwd())

In [ ]:
!pip install -q -r requirements.txt && pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# Unit tests (about a minute): text normalisation, features, data pipeline, training loop, LM, metrics.
!python -m pytest -q

## 2. Prepare MSA (Common Voice 16.1) and run the E0 baselines

In [ ]:
!python scripts/prepare_data.py --config configs/data.yaml --sources msa

In [ ]:
# E0a: zero-shot Whisper-small and the paper's MSA model (otozz/whisper-small-ar_tsize_1.0).
# Expected at the raw level (paper): 61.8% and ~45.9% WER. Check this before training anything.
!python scripts/evaluate.py --model openai/whisper-small --name zeroshot_small --test-sets msa --out $RESULTS/e0_zeroshot_small
!python scripts/evaluate.py --model otozz/whisper-small-ar_tsize_1.0 --name paper_msa --test-sets msa --out $RESULTS/e0_paper_msa

## 3. E1: fine-tune on MSA (resumable; re-run this cell after a disconnect)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $ASR_RUNS

In [ ]:
!python scripts/train.py --config configs/e1_msa.yaml

In [ ]:
# Quick check on 1,000 MSA test utterances (the full evaluation on all test sets follows in section 4).
!python scripts/evaluate.py --model $ASR_RUNS/e1_msa/best --name e1_msa_quick --test-sets msa --max-samples 1000 --out $RESULTS/e1_msa_quick

## 4. Prepare the five MASC dialects and evaluate the paper's pooled model (E0b)

In [ ]:
!python scripts/prepare_data.py --config configs/data.yaml --sources egyptian gulf iraqi levantine maghrebi

In [ ]:
!python scripts/evaluate.py --model otozz/whisper-small-finetune_all_seed42 --name paper_pooled --out $RESULTS/e0_paper_pooled
!python scripts/evaluate.py --model $ASR_RUNS/e1_msa/best --name e1_msa --out $RESULTS/e1_msa

## 5. E2: dialect-pooled model from the E1 checkpoint (main model)

In [ ]:
!python scripts/train.py --config configs/e2_pooled.yaml

In [ ]:
!python scripts/evaluate.py --model $ASR_RUNS/e2_pooled/best --name e2_pooled --out $RESULTS/e2_pooled

## 6. Language model: train on training transcripts, tune on validation, test with beam search

In [ ]:
!python scripts/train_lm.py --config configs/e2_pooled.yaml --out $ASR_RUNS/lm/arabic_3gram.pkl.gz
!python scripts/tune_lm.py --config configs/e2_pooled.yaml --model $ASR_RUNS/e2_pooled/best --lm $ASR_RUNS/lm/arabic_3gram.pkl.gz --num-beams 5 --max-samples 300 --out $RESULTS/lm_tuning.json
!cat $RESULTS/lm_tuning.json | head -20

In [ ]:
import json
best = json.load(open(f"{RESULTS}/lm_tuning.json"))["best"]
LMW, WB = best["lm_weight"], best["word_bonus"]
print(LMW, WB)
# Same random subset of 500 utterances per test set for both runs, so they are paired.
!python scripts/evaluate.py --model $ASR_RUNS/e2_pooled/best --name e2_beam5 --num-beams 5 --max-samples 500 --out $RESULTS/e2_beam5
!python scripts/evaluate.py --model $ASR_RUNS/e2_pooled/best --name e2_beam5_lm --num-beams 5 --n-best 5 --lm $ASR_RUNS/lm/arabic_3gram.pkl.gz --lm-weight {LMW} --word-bonus {WB} --max-samples 500 --out $RESULTS/e2_beam5_lm

## 7. Speed: fine-tuned small vs. large-v3 (paper: 0.12 s vs. 0.29 s per utterance on an A100)

In [ ]:
!python scripts/benchmark_speed.py --models $ASR_RUNS/e1_msa/best openai/whisper-large-v3 --n 200 --out $RESULTS/speed_benchmark.json

## 8. Report: tables, figures, significance tests

In [ ]:
!python scripts/analyze.py $RESULTS/e0_paper_pooled $RESULTS/e1_msa $RESULTS/e2_pooled --primary e2_pooled --compare paper_pooled e2_pooled --out $RESULTS/report_main
!python scripts/analyze.py $RESULTS/e0_zeroshot_small $RESULTS/e0_paper_msa $RESULTS/e1_msa --primary e1_msa --compare paper_msa e1_msa --out $RESULTS/report_msa
!python scripts/analyze.py $RESULTS/e2_beam5 $RESULTS/e2_beam5_lm --primary e2_beam5_lm --compare e2_beam5 e2_beam5_lm --out $RESULTS/report_lm
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/report_main/report.md", encoding="utf-8").read()))

## 9. (Optional) E3: Egyptian-only model, for dialect-specific vs. pooled

In [ ]:
!python scripts/train.py --config configs/e3_egyptian.yaml
!python scripts/evaluate.py --model $ASR_RUNS/e3_egyptian/best --name e3_egyptian --out $RESULTS/e3_egyptian
!python scripts/analyze.py $RESULTS/e2_pooled $RESULTS/e3_egyptian --primary e3_egyptian --compare e2_pooled e3_egyptian --out $RESULTS/report_specific_vs_pooled

## 10. Publish the trained weights (Hugging Face Hub, private by default)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
assert HF_USERNAME, "set HF_USERNAME in section 0"
!python scripts/push_to_hub.py $ASR_RUNS/e2_pooled/best {HF_USERNAME}/whisper-small-ar-dialects
!python scripts/push_to_hub.py $ASR_RUNS/e1_msa/best {HF_USERNAME}/whisper-small-ar-msa